# 🏆 Lichess Titled Players Beaten Tracker

Track how many titled players you've beaten on Lichess!

## 🚀 How to Use
1. **Run all cells** (Cell → Run All or Shift+Enter on each cell)
2. **Enter your Lichess username** in the form below
3. **Select time control** (All, Bullet, Blitz, etc.)
4. **Choose game scope** (start with 500 games)
5. **Click "Analyze Games"** and wait for results
6. **Download CSV** for detailed game data

## ✨ Features
- 🎯 **Counts titled players beaten**: GM, IM, FM, CM, WGM, WIM, WFM, WCM, NM, LM
- 📊 **Unique opponents**: See distinct titled players vs total wins
- 📈 **Rich visualizations**: Bar charts, rating histograms, timelines, time control breakdown
- 🔄 **Comparison mode**: Compare two players side-by-side
- 💾 **Auto-export CSV**: Detailed results saved automatically
- 🔁 **Retry logic**: Automatic retries with backoff on API errors
- 🆓 **Completely free**: No registration or tokens required


In [1]:
import subprocess
import sys

def install_package(package):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except Exception:
        print(f"⚠️ Failed to install {package}")

# Check and install required packages
required_packages = ['requests', 'pandas', 'ipywidgets', 'matplotlib']
for package in required_packages:
    try:
        __import__(package)
    except ImportError:
        print(f"📦 Installing {package}...")
        install_package(package)

# Import all required modules
import requests
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import json
from datetime import datetime
from collections import Counter
import time as time_module

print("✅ All packages loaded successfully!")

✅ All packages loaded successfully!


In [2]:
class LichessTitledPlayersTracker:
    """Track titled players beaten on Lichess"""

    def __init__(self):
        self.base_url = "https://lichess.org/api"
        self.session = requests.Session()

        # Set up headers for free API access
        self.session.headers.update({
            'User-Agent': 'Lichess Titled Players Tracker v2.0',
            'Accept': 'application/x-ndjson'
        })

        # All possible titles on Lichess
        self.titles = ['GM', 'IM', 'FM', 'CM', 'WGM', 'WIM', 'WFM', 'WCM', 'NM', 'LM']

        # Time control mappings
        self.time_controls = {
            'bullet': 'bullet',
            'blitz': 'blitz',
            'rapid': 'rapid',
            'classical': 'classical',
            'correspondence': 'correspondence'
        }

    def get_user_games(self, username, time_control=None, max_games=None,
                       progress_bar=None, max_retries=3):
        """Fetch games for a user from Lichess API with progress tracking and retry logic"""
        url = f"{self.base_url}/games/user/{username}"

        params = {
            'rated': 'true',
            'format': 'json'
        }

        if max_games:
            params['max'] = max_games

        if time_control and time_control != 'all':
            params['perfType'] = time_control

        scope_text = f"last {max_games}" if max_games else "all"
        tc_text = time_control if time_control and time_control != 'all' else "all time controls"

        # Retry loop with exponential backoff
        for attempt in range(1, max_retries + 1):
            try:
                if attempt == 1:
                    print(f"🔍 Fetching {scope_text} {tc_text} games for {username}...")
                else:
                    wait_time = 2 ** attempt  # 4s, 8s
                    print(f"🔄 Retry {attempt}/{max_retries} in {wait_time}s...")
                    time_module.sleep(wait_time)
                    print(f"🔍 Retrying fetch for {username}...")

                response = self.session.get(url, params=params, stream=True, timeout=60)

                # Handle rate limiting (429)
                if response.status_code == 429:
                    retry_after = int(response.headers.get('Retry-After', 60))
                    print(f"⏳ Rate limited. Waiting {retry_after}s...")
                    time_module.sleep(retry_after)
                    continue

                response.raise_for_status()

                games = []
                # Set up progress bar
                if progress_bar is not None:
                    progress_bar.max = max_games if max_games else 1000
                    progress_bar.value = 0
                    progress_bar.description = "Fetching: "
                    progress_bar.layout.visibility = 'visible'

                for line in response.iter_lines():
                    if line:
                        try:
                            game = json.loads(line.decode('utf-8'))
                            games.append(game)

                            # Update progress bar
                            if progress_bar is not None:
                                progress_bar.value = len(games)
                                # Auto-expand max if we exceed it (for "all games")
                                if len(games) >= progress_bar.max:
                                    progress_bar.max = len(games) + 500
                                progress_bar.description = f"Fetching: {len(games)} "

                        except json.JSONDecodeError:
                            continue

                # Finalize progress bar
                if progress_bar is not None:
                    progress_bar.max = len(games)
                    progress_bar.value = len(games)
                    progress_bar.description = "Done! "
                    progress_bar.bar_style = 'success'

                print(f"✅ Fetched {len(games)} games")
                return games

            except requests.exceptions.ConnectionError:
                print(f"❌ Connection error (attempt {attempt}/{max_retries})")
            except requests.exceptions.Timeout:
                print(f"❌ Request timed out (attempt {attempt}/{max_retries})")
            except requests.exceptions.RequestException as e:
                print(f"❌ Error fetching games: {e}")
                if progress_bar is not None:
                    progress_bar.bar_style = 'danger'
                    progress_bar.description = "Error! "
                return []  # Non-retryable error

        # All retries exhausted
        print(f"❌ Failed after {max_retries} attempts. Please try again later.")
        if progress_bar is not None:
            progress_bar.bar_style = 'danger'
            progress_bar.description = "Failed! "
        return []

    def analyze_titled_opponents(self, username, games):
        """Analyze games to find titled opponents that were beaten"""
        titled_beaten = {title: [] for title in self.titles}
        total_games_analyzed = 0
        wins_against_titled = 0

        for game in games:
            total_games_analyzed += 1

            # Get player data
            players = game.get('players', {})
            white_player = players.get('white', {})
            black_player = players.get('black', {})

            # Determine user's color and opponent
            white_name = white_player.get('user', {}).get('name', '').lower()
            black_name = black_player.get('user', {}).get('name', '').lower()

            if white_name == username.lower():
                user_color = 'white'
                opponent_data = black_player
            elif black_name == username.lower():
                user_color = 'black'
                opponent_data = white_player
            else:
                continue

            # Check if user won
            winner = game.get('winner')
            if winner != user_color:
                continue

            # Check if opponent has a title
            opponent_user = opponent_data.get('user', {})
            opponent_title = opponent_user.get('title')
            opponent_username = opponent_user.get('name', 'Unknown')

            if opponent_title and opponent_title in self.titles:
                wins_against_titled += 1

                # Get time control (perf field is a string like 'bullet', 'blitz')
                time_control_name = game.get('perf', 'Unknown')

                game_info = {
                    'opponent': opponent_username,
                    'date': datetime.fromtimestamp(game.get('createdAt', 0) / 1000).strftime('%Y-%m-%d'),
                    'time_control': time_control_name.title() if time_control_name else 'Unknown',
                    'rating': opponent_data.get('rating', 'N/A'),
                    'game_url': f"https://lichess.org/{game.get('id', '')}"
                }
                titled_beaten[opponent_title].append(game_info)

        print(f"📊 Analyzed {total_games_analyzed} games, found {wins_against_titled} wins against titled players")
        return titled_beaten

print("🚀 Tracker class loaded successfully!")

🚀 Tracker class loaded successfully!


In [3]:
# 🎮 MAIN APPLICATION - Clean Interface with Comparison Mode

# ===== Color scheme =====
TITLE_COLORS = {
    'GM': '#FFD700', 'IM': '#C0C0C0', 'FM': '#CD7F32', 'CM': '#8B7355',
    'WGM': '#FF69B4', 'WIM': '#DDA0DD', 'WFM': '#DA70D6', 'WCM': '#BA55D3',
    'NM': '#87CEEB', 'LM': '#98FB98'
}
TC_COLORS = {
    'Bullet': '#e74c3c', 'Blitz': '#f39c12', 'Rapid': '#2ecc71',
    'Classical': '#3498db', 'Correspondence': '#9b59b6', 'Unknown': '#95a5a6'
}
ALL_TITLES = ['GM', 'IM', 'FM', 'CM', 'WGM', 'WIM', 'WFM', 'WCM', 'NM', 'LM']

# ===== Widget inputs =====
username_input = widgets.Text(
    value='',
    placeholder='Enter your Lichess username',
    description='Username 1:',
    style={'description_width': 'initial'},
    layout={'width': '350px'}
)

# Comparison mode username
username_input_2 = widgets.Text(
    value='',
    placeholder='Leave empty for single analysis',
    description='Username 2:',
    style={'description_width': 'initial'},
    layout={'width': '350px'}
)

time_control_dropdown = widgets.Dropdown(
    options=[
        ('All time controls', 'all'),
        ('Bullet', 'bullet'),
        ('Blitz', 'blitz'),
        ('Rapid', 'rapid'),
        ('Classical', 'classical'),
        ('Correspondence', 'correspondence')
    ],
    value='all',
    description='Time Control:',
    style={'description_width': 'initial'}
)

game_scope_dropdown = widgets.Dropdown(
    options=[
        ('All games (complete history)', 'all'),
        ('Recent 100 games (quick test)', 100),
        ('Recent 500 games', 500),
        ('Recent 1000 games', 1000),
        ('Recent 2000 games', 2000),
        ('Recent 5000 games', 5000)
    ],
    value=500,
    description='Game Scope:',
    style={'description_width': 'initial'}
)

analyze_button = widgets.Button(
    description='🔍 Analyze Games',
    button_style='primary',
    icon='search',
    layout={'width': '200px', 'height': '40px'}
)

progress_bar = widgets.IntProgress(
    value=0, min=0, max=100,
    description='Ready',
    bar_style='info',
    style={'bar_color': '#4CAF50', 'description_width': 'initial'},
    layout={'width': '400px', 'visibility': 'hidden'}
)

output_area = widgets.Output()
tracker = None


# ========== HELPER: get unique + total stats ==========
def get_stats(titled_beaten):
    """Return (total_wins, unique_opponents, per-title breakdown) from titled_beaten dict."""
    total_wins = 0
    unique_opponents = set()
    title_stats = {}  # {title: {'wins': int, 'unique': int}}

    for title in ALL_TITLES:
        games = titled_beaten[title]
        wins = len(games)
        unique = len(set(g['opponent'] for g in games))
        total_wins += wins
        unique_opponents.update(g['opponent'] for g in games)
        if wins > 0:
            title_stats[title] = {'wins': wins, 'unique': unique}

    return total_wins, len(unique_opponents), title_stats


# ========== VISUALIZATIONS ==========
def create_visualizations(titled_beaten, username):
    """Create 4 charts: bar by title, rating histogram, timeline, time control breakdown"""
    # Collect data
    title_counts = {}
    all_ratings = []
    all_dates = []
    tc_counter = Counter()

    for title in ALL_TITLES:
        count = len(titled_beaten[title])
        if count > 0:
            title_counts[title] = count
            for g in titled_beaten[title]:
                if isinstance(g['rating'], (int, float)):
                    all_ratings.append(g['rating'])
                try:
                    all_dates.append(datetime.strptime(g['date'], '%Y-%m-%d'))
                except ValueError:
                    pass
                tc_counter[g['time_control']] += 1

    if not title_counts:
        return

    plt.style.use('seaborn-v0_8-darkgrid')
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"{username}'s Titled Scalps", fontsize=17, fontweight='bold', y=1.01)

    # --- Chart 1: Bar chart of titles beaten ---
    ax1 = axes[0, 0]
    titles = list(title_counts.keys())
    counts = list(title_counts.values())
    colors = [TITLE_COLORS.get(t, '#4CAF50') for t in titles]

    bars = ax1.bar(titles, counts, color=colors, edgecolor='white', linewidth=1.2)
    for bar, c in zip(bars, counts):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                 str(c), ha='center', va='bottom', fontweight='bold', fontsize=11)
    ax1.set_title('Wins by Title', fontsize=13, fontweight='bold')
    ax1.set_ylabel('Wins', fontsize=11)
    ax1.set_ylim(0, max(counts) * 1.25)

    # --- Chart 2: Rating distribution ---
    ax2 = axes[0, 1]
    if all_ratings:
        ax2.hist(all_ratings, bins=15, color='#4CAF50', edgecolor='white', alpha=0.85)
        avg_r = sum(all_ratings) / len(all_ratings)
        ax2.axvline(x=avg_r, color='#FF5722', linestyle='--', linewidth=2,
                    label=f'Avg: {avg_r:.0f}')
        ax2.legend(fontsize=11, framealpha=0.9)
    ax2.set_title('Opponent Rating Distribution', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Rating', fontsize=11)
    ax2.set_ylabel('Frequency', fontsize=11)

    # --- Chart 3: Win timeline ---
    ax3 = axes[1, 0]
    if all_dates:
        all_dates_sorted = sorted(all_dates)
        cumulative = list(range(1, len(all_dates_sorted) + 1))
        ax3.plot(all_dates_sorted, cumulative, color='#2196F3', linewidth=2.5, marker='o',
                 markersize=4, markerfacecolor='#FF9800', markeredgecolor='white')
        ax3.fill_between(all_dates_sorted, cumulative, alpha=0.15, color='#2196F3')
        ax3.xaxis.set_major_formatter(DateFormatter('%b %Y'))
        plt.setp(ax3.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax3.set_title('Titled Wins Over Time', fontsize=13, fontweight='bold')
    ax3.set_xlabel('Date', fontsize=11)
    ax3.set_ylabel('Cumulative Wins', fontsize=11)

    # --- Chart 4: Time control breakdown (pie) ---
    ax4 = axes[1, 1]
    if tc_counter:
        tc_labels = list(tc_counter.keys())
        tc_values = list(tc_counter.values())
        tc_colors = [TC_COLORS.get(label, '#95a5a6') for label in tc_labels]
        wedges, texts, autotexts = ax4.pie(
            tc_values, labels=tc_labels, colors=tc_colors, autopct='%1.0f%%',
            startangle=90, textprops={'fontsize': 11})
        for at in autotexts:
            at.set_fontweight('bold')
    ax4.set_title('Wins by Time Control', fontsize=13, fontweight='bold')

    plt.tight_layout()
    display(fig)
    plt.close(fig)


def create_comparison_chart(stats1, stats2, user1, user2):
    """Side-by-side bar chart comparing two users"""
    import numpy as np

    # Get union of titles both users have
    all_present_titles = sorted(
        set(list(stats1.keys()) + list(stats2.keys())),
        key=lambda t: ALL_TITLES.index(t) if t in ALL_TITLES else 99
    )

    if not all_present_titles:
        print("No titled wins to compare.")
        return

    plt.style.use('seaborn-v0_8-darkgrid')
    fig, ax = plt.subplots(figsize=(12, 6))

    x = np.arange(len(all_present_titles))
    width = 0.35

    vals1 = [stats1.get(t, {}).get('wins', 0) for t in all_present_titles]
    vals2 = [stats2.get(t, {}).get('wins', 0) for t in all_present_titles]

    bars1 = ax.bar(x - width / 2, vals1, width, label=user1, color='#2196F3',
                   edgecolor='white', linewidth=1.2)
    bars2 = ax.bar(x + width / 2, vals2, width, label=user2, color='#FF9800',
                   edgecolor='white', linewidth=1.2)

    # Value labels
    for bar in bars1:
        if bar.get_height() > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                    str(int(bar.get_height())), ha='center', fontweight='bold', fontsize=10)
    for bar in bars2:
        if bar.get_height() > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                    str(int(bar.get_height())), ha='center', fontweight='bold', fontsize=10)

    ax.set_xticks(x)
    ax.set_xticklabels(all_present_titles, fontsize=11)
    ax.set_ylabel('Wins', fontsize=12)
    ax.set_title(f'{user1} vs {user2} - Titled Wins Comparison',
                 fontsize=15, fontweight='bold')
    ax.legend(fontsize=12, framealpha=0.9)
    max_val = max(max(vals1, default=0), max(vals2, default=0))
    ax.set_ylim(0, max_val * 1.3 if max_val > 0 else 1)

    plt.tight_layout()
    display(fig)
    plt.close(fig)


# ========== DISPLAY RESULTS ==========
def display_clean_results(titled_beaten, username, is_comparison=False):
    """Display clean summary with unique opponent stats, visualizations, and CSV export"""
    total_wins, unique_count, title_stats = get_stats(titled_beaten)

    print("\n" + "=" * 55)
    print(f"🏆 {username.upper()} vs TITLED PLAYERS")
    print("=" * 55)

    if total_wins > 0:
        # Summary with unique opponents
        print(f"\n📊 RESULTS (wins / unique opponents):")
        for title in ALL_TITLES:
            if title in title_stats:
                s = title_stats[title]
                unique_tag = f"  ({s['unique']} unique)" if s['unique'] != s['wins'] else ""
                print(f"  {title}: {s['wins']}{unique_tag}")

        print(f"\n🎉 TOTAL: {total_wins} wins against {unique_count} unique titled players!")

        # Build & export CSV
        results_data = []
        for title in ALL_TITLES:
            for g in titled_beaten[title]:
                results_data.append({
                    'Title': title, 'Opponent': g['opponent'],
                    'Rating': g['rating'], 'Date': g['date'],
                    'Time_Control': g['time_control'], 'Game_URL': g['game_url']
                })

        df = pd.DataFrame(results_data)
        filename = f"{username}_titled_players.csv"
        df.to_csv(filename, index=False)
        print(f"\n💾 Results saved to: {filename}")
        print(f"📁 Contains {len(results_data)} games with full details")

        print("\n📋 Preview (first 5 games):")
        display(df[['Title', 'Opponent', 'Rating', 'Date', 'Time_Control']].head(5))

        # Full visualizations (skip if comparison — comparison chart shown separately)
        if not is_comparison:
            print("\n📈 VISUALIZATIONS:")
            create_visualizations(titled_beaten, username)

    else:
        print("\n😔 No titled players beaten in analyzed games")
        print("💡 Try: larger game scope, different time control")
        print("🎯 Keep playing - beating titled players is rare!")

    return total_wins, unique_count, title_stats


# ========== MAIN ANALYSIS ==========
def run_analysis(username, time_control, game_scope, show_progress=True):
    """Run analysis for a single user. Returns (titled_beaten, tracker) or None."""
    t = LichessTitledPlayersTracker()

    scope_text = "ALL games" if game_scope == 'all' else f"recent {game_scope} games"
    print(f"🎯 Analyzing {scope_text} for {username}")
    print(f"⏱️ Time control: {time_control if time_control != 'all' else 'All time controls'}")
    print("-" * 55)

    max_games = None if game_scope == 'all' else game_scope
    pbar = progress_bar if show_progress else None
    games = t.get_user_games(
        username,
        time_control if time_control != 'all' else None,
        max_games,
        progress_bar=pbar
    )

    if not games:
        print("❌ No games found")
        print("💡 Check username spelling and try again")
        return None, t

    print("\n🎯 Analyzing for titled opponents...")
    titled_beaten = t.analyze_titled_opponents(username, games)
    return titled_beaten, t


def analyze_games(button):
    """Main analysis function — single or comparison mode"""
    global tracker

    with output_area:
        clear_output(wait=True)

        username1 = username_input.value.strip()
        username2 = username_input_2.value.strip()

        if not username1:
            print("❌ Please enter at least Username 1")
            return

        # Reset progress bar
        progress_bar.value = 0
        progress_bar.bar_style = 'info'
        progress_bar.style = {'bar_color': '#4CAF50', 'description_width': 'initial'}
        progress_bar.description = 'Starting...'
        progress_bar.layout.visibility = 'visible'

        tc = time_control_dropdown.value
        scope = game_scope_dropdown.value
        is_compare = bool(username2)

        # ===== Analyze user 1 =====
        titled1, tracker = run_analysis(username1, tc, scope)
        if titled1 is None:
            return
        _, _, stats1 = display_clean_results(titled1, username1, is_comparison=is_compare)

        # ===== Analyze user 2 (if comparison mode) =====
        if is_compare:
            print("\n\n" + "🔄" * 25)
            print(f"\n⏳ Now analyzing {username2}...\n")

            # Reset progress bar for user 2
            progress_bar.value = 0
            progress_bar.bar_style = 'info'
            progress_bar.style = {'bar_color': '#FF9800', 'description_width': 'initial'}
            progress_bar.description = 'Starting...'

            titled2, _ = run_analysis(username2, tc, scope)
            if titled2 is None:
                return
            _, _, stats2 = display_clean_results(titled2, username2, is_comparison=True)

            # Side-by-side comparison chart
            print("\n\n" + "=" * 55)
            print("📊 HEAD-TO-HEAD COMPARISON")
            print("=" * 55)

            total1 = sum(s['wins'] for s in stats1.values())
            total2 = sum(s['wins'] for s in stats2.values())
            unique1 = len(set(g['opponent'] for t in ALL_TITLES for g in titled1[t]))
            unique2 = len(set(g['opponent'] for t in ALL_TITLES for g in titled2[t]))

            print(f"\n  {'':20s} {username1:>15s}  vs  {username2:<15s}")
            print(f"  {'Total wins':20s} {total1:>15d}       {total2:<15d}")
            print(f"  {'Unique opponents':20s} {unique1:>15d}       {unique2:<15d}")

            for title in ALL_TITLES:
                w1 = stats1.get(title, {}).get('wins', 0)
                w2 = stats2.get(title, {}).get('wins', 0)
                if w1 > 0 or w2 > 0:
                    print(f"  {title:20s} {w1:>15d}       {w2:<15d}")

            create_comparison_chart(stats1, stats2, username1, username2)

            # Individual charts for each user
            print(f"\n📈 {username1}'s Charts:")
            create_visualizations(titled1, username1)
            print(f"\n📈 {username2}'s Charts:")
            create_visualizations(titled2, username2)


# Connect button
analyze_button.on_click(analyze_games)

# Display UI
display(HTML("<h2>🏆 Lichess Titled Players Tracker</h2>"))
display(HTML("<p><strong>Enter your details and click analyze! Add a 2nd username to compare.</strong></p>"))

display(widgets.VBox([
    username_input,
    username_input_2,
    time_control_dropdown,
    game_scope_dropdown,
    widgets.HTML('<br>'),
    analyze_button,
    progress_bar,
    widgets.HTML('<br>'),
    output_area
]))

print("\n🎮 Ready! Fill in your details above and click 'Analyze Games'")
print("📊 Single user: full analysis + 4 charts + CSV export")
print("🔄 Two users: side-by-side comparison + individual charts")


🎮 Ready! Fill in your details above and click 'Analyze Games'
📊 Single user: full analysis + 4 charts + CSV export
🔄 Two users: side-by-side comparison + individual charts
